# Structure Analysis: IO, Structure graph and Pattern matching


## Imports
Let's set some paths and import mofography, 

In [ ]:
#* make imports
import mofography as mgr

# CIFs ship with the repo (docs/ is the working directory)
path_to_cif = '../examples/TestCIFs/P1-IRMOF-1.cif'

## File IO
Mofography uses ASE as a backend for structure IO and manipulation. Hence, all the formats suppotred by ASE are supported by Mofography. The following example shows how to read in a structure from a CIF file.



In [2]:

from ase.io import read 
mat_atoms = read('../examples/TestCIFs/LTA.cif')

c:\Users\Public\anaconda3\envs\test\lib\site-packages\ase\io\cif.py:401: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(221, setting=1). This may result in wrong setting!
  warnings.warn(


mofography also provides a fast Rust CIF reader, `from mofography.cif_io import read_cif_to_ase`, which returns the same `ase.Atoms` and is much faster for large structures and many files. See [Fast Rust CIF reader](fast_cif_reader.qmd).

if for some formatting reason, ASE cannot read in your file, you can use pymatgen to read in the structure and convert it to an ASE Atoms object as shown below

In [3]:
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

mat_atoms = AseAtomsAdaptor.get_atoms(Structure.from_file('../examples/TestCIFs/P1-IRMOF-1.cif'))

C:\Users\Arun Gopalan\AppData\Roaming\Python\Python39\site-packages\pymatgen\io\cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(


## Covalent Bonds
mofography provdes a shorthand subroutine for detecing the covalent bonds in the structure, which quite useful for analysis and visualization. By default, it looks for atomic distances that are lesser than the sum of the covalent radii of the two atoms and considers that a bond. The following example shows how to detect the bonds in a structure using mofography

In [4]:


mat_updated = mgr.update_with_covalent_bonds(mat_atoms)

the detected bonds and bond orders, will stored under the 'covalent_bonds/_bond_orders' keys of the mat_updated.info dictionary. The following example shows how to access them, just showing the first 10 entries here

In [5]:
mat_updated.info['covalent_bonds'][:10], mat_updated.info['covalent_bond_orders'][:10]

([[0, 320],
  [0, 328],
  [0, 345],
  [0, 352],
  [1, 323],
  [1, 329],
  [1, 344],
  [1, 415],
  [2, 321],
  [2, 330]],
 [0.8355920148601726,
  0.8886078731209055,
  0.888607873120906,
  0.8886078731209055,
  0.8355920148601627,
  0.888607873120912,
  0.8886078731209055,
  0.8886078731209014,
  0.8355920148601726,
  0.8886078731209044])

## Structure graph

Structure graph is networkx graph object, formed using atoms as nodes and bonds as edges. Mofography offers a simple routine to generate a strcture graph from an ASE atoms object. 
- Having a structure graph is quite useful for analysing the structure at hand, such as detecting chemical moeties such functional groups, rings (aromatic and otherwise). 
- Annotating the atoms and bonds with relevant information, to be used in conjuction with other routines or tools.
Sgraph is a simpler version of the pymatgen structure graph, put in place to perform complex structure manipulations in the future.

In [6]:
mat_sgraph = mgr.sgraph_from_atoms(mat_updated)

No node props provided, using default set: [position, scaled_position, element and atomic_number]
No edge properties provided. Use the default set: [bond_order, bond_type]


by default it makes a graph with just the positions, element and atomic numbers of the atoms as node attributes, and the bond orders as edge attributes. 

In [7]:
mat_sgraph.nodes[0]

{'position': array([7.5791088, 5.3368912, 5.3368912]),
 'element': 'Zn',
 'scaled_position': array([0.2934, 0.2066, 0.2066]),
 'atomic_number': 30}

In [8]:
mat_sgraph.edges([0,192],data=True)

EdgeDataView([(0, 320, {'bond_order': 0.8355920148601726, 'bond_type': 'single'}), (0, 328, {'bond_order': 0.8886078731209055, 'bond_type': 'single'}), (0, 345, {'bond_order': 0.888607873120906, 'bond_type': 'single'}), (0, 352, {'bond_order': 0.8886078731209055, 'bond_type': 'single'}), (192, 96, {'bond_order': 1, 'bond_type': 'single'}), (192, 456, {'bond_order': 1.7511920000000007, 'bond_type': 'double'}), (192, 285, {'bond_order': 1.7584099829932225, 'bond_type': 'double'})])

## Finding rings in a structure 

Further, we can use a networkx backend to find cycles (rings) in a structure. These include aromatic and non-aromatic rings. Finding rings can be very useful in systems such as zeolites, where such rings represent the cage windows are often the preferred sites for cations. The following example shows how to find the 6 membered rings in a structure using networkx

In [9]:
import networkx as nx
import numpy as np

default_ring_sizes = [6]
small_cycles = [cycle for cycle in nx.cycles.cycle_basis(mat_sgraph, 0) if len(cycle) in default_ring_sizes]
small_cycles

[[320, 7, 348, 256, 352, 0],
 [320, 4, 338, 240, 341, 7],
 [320, 9, 357, 266, 354, 4],
 [326, 18, 361, 278, 389, 27],
 [326, 24, 370, 292, 403, 18],
 [326, 21, 365, 284, 392, 24],
 [321, 2, 422, 246, 342, 6],
 [321, 15, 355, 268, 412, 2],
 [321, 11, 343, 248, 418, 15],
 [325, 26, 381, 310, 399, 25],
 [325, 17, 360, 276, 388, 26],
 [322, 5, 423, 242, 339, 3],
 [322, 13, 350, 260, 416, 5],
 [322, 12, 346, 254, 420, 13],
 [324, 16, 402, 302, 377, 22],
 [324, 30, 378, 304, 397, 16],
 [324, 19, 368, 288, 405, 30],
 [164, 135, 237, 137, 167, 283],
 [323, 8, 419, 250, 344, 1],
 [323, 14, 353, 264, 414, 8],
 [323, 10, 340, 244, 421, 14],
 [327, 29, 387, 318, 395, 23],
 [327, 20, 363, 280, 391, 29],
 [327, 28, 380, 308, 398, 20],
 [129, 190, 275, 159, 208, 227],
 [177, 215, 259, 149, 202, 303],
 [139, 196, 315, 184, 223, 243],
 [171, 213, 271, 156, 204, 295],
 [325, 31, 385, 316, 394, 17],
 [134, 193, 287, 166, 210, 235],
 [174, 217, 263, 151, 201, 299],
 [145, 179, 307, 182, 147, 253]]

In [10]:
nx.subgraph(mat_sgraph, small_cycles[0]).nodes('element')

NodeDataView({320: 'O', 256: 'C', 352: 'O', 0: 'Zn', 7: 'Zn', 348: 'O'}, data='element')

## Find matching patterns in the structure

mofography uses subgraph isomorphism as implemented in networkx to match parts of a structure to a given pattern. For example if we want to find all the occurences of the first cycle from the previous step to everywhere else in the framework, we can use the following code, 

In [11]:
matching = mgr.find_pattern_in_sgraph(mat_sgraph, nx.subgraph(mat_sgraph,small_cycles[0]))

Now if we check the atom and bond types of the first subgraph where the match is found, 

In [12]:

first_match_graph = nx.subgraph(mat_sgraph, matching[0])
first_match_graph.nodes(data='element'), first_match_graph.edges(data='bond_type')

(NodeDataView({320: 'O', 0: 'Zn', 352: 'O', 256: 'C', 7: 'Zn', 348: 'O'}, data='element'),
 EdgeDataView([(320, 0, 'single'), (320, 7, 'single'), (0, 352, 'single'), (352, 256, 'double'), (256, 348, 'double'), (7, 348, 'single')]))

## Where to next?

There are a few options for you to jump to from here, for example the tutorial on Quick Structure Visualization would be very handy to visualize things in 3D in a jupyter notebook or your browser. Another, option is the tutorial on Calculating Distance Grid, as this would be the first step in the core mofography workflow for analyzing any structure.